# 04 · Global similarity between protein representations

Do the pLMs place viral and human LxCxE peptides close together or apart?
For each model, the WT peptide embeddings (positions × embedding dimension) of the five proteins
are compared pairwise with the **Pearson correlation** (report, section 2, *Global similarity between
protein representations*) and, as a complement, the cosine similarity.

**Input:** `data/processed/df_tp.csv`, `data/processed/wt_embeddings.npz` (notebooks 01 and 03).
**Output:** `results/protein_similarity_<metric>_<model>.csv` and figures.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Project paths
#   * Locally: start Jupyter from the `notebooks/` folder.
#   * Google Colab: Google Drive is mounted; edit the folder name below so it
#     points to the project folder stored in your Drive.
#   * Anywhere: set the PROJECT_DIR environment variable to override.
# ---------------------------------------------------------------------------
try:
    from google.colab import drive

    drive.mount("/content/drive")
    DEFAULT_PROJECT_DIR = Path("/content/drive/MyDrive/plm-embeddings-lxcxe-prb")
except ImportError:
    DEFAULT_PROJECT_DIR = Path.cwd().parent

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", DEFAULT_PROJECT_DIR))
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"
for folder in (PROCESSED_DIR, RESULTS_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def save_figure(name):
    """Save the current matplotlib figure into figures/ (call before plt.show())."""
    plt.savefig(FIGURES_DIR / name, dpi=300, bbox_inches="tight")

import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load WT embeddings

In [ ]:
df_tp = pd.read_csv(PROCESSED_DIR / "df_tp.csv")
PROTEINS = df_tp["protein"].tolist()

with np.load(PROCESSED_DIR / "wt_embeddings.npz") as archive:
    wt_embeddings = {
        tuple(key.split("__", 1)): archive[key] for key in archive.files
    }  # (protein, model) -> array (peptide length, dim)

MODELS = sorted({model for _, model in wt_embeddings})
example = wt_embeddings[(PROTEINS[0], MODELS[0])]
print(f"Models: {MODELS} | e.g. embedding matrix shape: {example.shape}")

## 2. Similarity matrices

Each protein is represented by its **full embedding matrix flattened into one vector**
(`peptide length × dim` values), so residues are compared position by position. As a robustness check,
`mode="mean"` first averages the embedding over positions (one `dim`-dimensional vector per peptide).

In [ ]:
def protein_vectors(model_name, mode="flatten"):
    """Rows = proteins, columns = embedding features."""
    if mode == "flatten":
        vectors = [wt_embeddings[(p, model_name)].ravel() for p in PROTEINS]
    elif mode == "mean":
        vectors = [wt_embeddings[(p, model_name)].mean(axis=0) for p in PROTEINS]
    else:
        raise ValueError("mode must be 'flatten' or 'mean'")
    return np.vstack(vectors)


def similarity_matrices(model_name, mode="flatten"):
    """Pearson and cosine similarity between the proteins (5 x 5 each)."""
    vectors = protein_vectors(model_name, mode)
    pearson = pd.DataFrame(np.corrcoef(vectors), index=PROTEINS, columns=PROTEINS)
    cosine = pd.DataFrame(cosine_similarity(vectors), index=PROTEINS, columns=PROTEINS)
    return pearson, cosine


def plot_similarity(matrix, title, vmin=-1, cmap="coolwarm", save_as=None):
    plt.figure(figsize=(7, 6))
    sns.heatmap(matrix, annot=True, fmt=".2f", cmap=cmap, vmin=vmin, vmax=1, square=True)
    plt.title(title)
    plt.tight_layout()
    if save_as:
        save_figure(save_as)
    plt.show()

In [ ]:
similarity = {}
for model_name in MODELS:
    pearson, cosine = similarity_matrices(model_name)
    similarity[model_name] = {"pearson": pearson, "cosine": cosine}

    slug = model_name.split(" ")[0].lower().replace("-", "")
    pearson.to_csv(RESULTS_DIR / f"protein_similarity_pearson_{slug}.csv")
    cosine.to_csv(RESULTS_DIR / f"protein_similarity_cosine_{slug}.csv")

    plot_similarity(pearson, f"Pearson correlation between proteins ({model_name})",
                    save_as=f"protein_pearson_{slug}.png")

## 3. Complement: cosine similarity and mean-pooled representation

In [ ]:
for model_name in MODELS:
    plot_similarity(similarity[model_name]["cosine"], f"Cosine similarity ({model_name})",
                    vmin=None, cmap="viridis")

In [ ]:
for model_name in MODELS:
    pearson_mean, _ = similarity_matrices(model_name, mode="mean")
    plot_similarity(pearson_mean, f"Pearson correlation, mean-pooled embeddings ({model_name})")